# 📝 Assignment — Chapter 8 — Linking Ontologies to Data — agentic lab

This is the **student** notebook. It contains 3 graded tasks.

Work top to bottom. Where you meet a **Task**, replace the `NotImplementedError` with your implementation and run the checks cell that follows. **The assertions in the checks cells are the marking scheme** — when they all pass, the assignment is complete.

Cells outside the tasks are worked examples. Read and run them: they build what the tasks need.

> Worked solutions: [`04_agentic_lab_solution.ipynb`](04_agentic_lab_solution.ipynb)

# Chapter 8 — Linking Ontologies to Data
### Notebook 4 · Agentic lab — mapping shapes and execution strategy

*Book reference: Extends §8.2–8.3*

An agent that makes the two decisions of this chapter, and an MDP in which the cheapest action is also the one that returns wrong answers.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch08_toolkit as ch8
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import ch08_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Give an agent a tool that checks two execution paths agree, and see why that beats any prompt instruction.
2. Stratify a dataset on **two independent dimensions** at once.
3. Model a workload as an MDP where staleness is lost **reward**, not added latency.
4. Read an optimal refresh policy and explain each decision economically.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

`compare_strategies` is the one that matters. An agent that calls it cannot ship the IRI/literal bug, regardless of what it believes about mappings — the distinction is enforced by a tool rather than asserted in a prompt.

In [ ]:
ctx = AG.Ch8Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:22s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":22s} {t.description.splitlines()[0]}')

In [ ]:
schema = json.loads(tools['inspect_schema'].invoke({}))
print('patient table:')
print('  columns     :', [c['name'] for c in schema['patient']['columns']])
print('  foreign keys:', schema['patient']['foreign_keys'])
print()
print(tools['compare_strategies'].invoke({'name': 'cardiac-patients-in-cardiology'}))
print(tools['run_query'].invoke({'name': 'patients-in-cardiology',
                                 'strategy': 'rewrite'}))
print('\ntrajectory:', ctx.log.names())

## 2. The dataset, stratified on two dimensions

Each case pairs a **column** (foreign key or attribute) with a **workload** (static or volatile), and the agent must get both right. Those are independent, so the split has to stratify on both — alternating over the raw list would have put every static workload in train and every volatile one in dev, leaving each half unable to teach one of the strategies.

In [ ]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print(pd.DataFrame([{'id': e.id, 'object_kind': e.gold_object_kind,
                     'strategy': e.gold_strategy,
                     'split': 'train' if e in train else 'dev'}
                    for e in AG.build_dataset('all')]).to_string(index=False))
print()
print('train combinations:', sorted({(e.gold_object_kind, e.gold_strategy) for e in train}))
print('dev   combinations:', sorted({(e.gold_object_kind, e.gold_strategy) for e in dev}))
assert ({(e.gold_object_kind, e.gold_strategy) for e in train}
        == {(e.gold_object_kind, e.gold_strategy) for e in dev})

## 3. Baseline and GEPA

The naive agent does what real projects do: emits literals unless told otherwise, and loads everything into a warehouse.

In [ ]:
lm = llm.configure_dspy(AG.OBDA_RULEBOOK, AG.obda_responder)
baseline = AG.OBDAProgram()
for e in dev:
    p = baseline(**e.inputs())
    print(f'{e.id:20s} answered {p.object_kind:8s}/{p.strategy:12s} '
          f'gold {e.gold_object_kind:8s}/{e.gold_strategy}')

In [ ]:
before = ev.evaluate_dataset(baseline, dev, AG.obda_scorer)
print('BEFORE:', before['mean_score'], before['violations'])

In [ ]:
gepa_metric = ev.make_gepa_metric(AG.obda_scorer, AG.OBDA_RULEBOOK)
reflect = llm.reflection_lm(AG.OBDA_RULEBOOK, AG.obda_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=80, reflection_lm=reflect)
result = opt.compare(AG.OBDAProgram(), tuned, dev, AG.obda_scorer)
print(result.report())

In [ ]:
found = AG.OBDA_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules not needed:', sorted(set(AG.OBDA_RULEBOOK.ids) - found))
print('\nThe two undiscovered rules describe the agent\'s DEFAULT behaviour --\n'
      'literals for attributes, materialise for static data. It was already\n'
      'doing those, so the metric never complained and there was nothing to\n'
      'learn. An undiscovered rule is not automatically a failure; check\n'
      'whether it was ever violated before concluding anything.')

## 4. The materialise-or-rewrite MDP

A known workload interleaves queries (`q`) with updates (`u`). An update makes any materialised copy stale.

| | |
|---|---|
| **S** | position in the workload, and whether the copy is fresh |
| **A** | serve by rewriting · serve from the copy · refresh, then serve |
| **T** | deterministic |
| **R** | `+1` for a **correct** answer, minus the cost of serving it |

The trap is deliberate. Serving from a stale copy is the **cheapest** action and earns **nothing**, because the answer is wrong. An agent optimising latency alone walks straight into it.

In [ ]:
M = AG.MaterialisationMDP(workload='qquqqquq')
print('workload:', M.workload, ' (q = query, u = update)')
print(f'costs: rewrite={M.cost_rewrite}, from copy={M.cost_materialised}, '
      f'refresh={M.cost_refresh}')
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'\nV*(s0) = {V[s0]:.3f}   (8 events, 6 of them queries)')

In [ ]:
state = s0
while not M.is_terminal(state):
    action = pi[state]
    print('  ' + M.describe(state, action))
    state = M.transition(state, action)[0][1]

> **Read the policy as economics.** The first two queries are served by rewriting — an update is coming, so a refresh would be wasted. After the update there are *three* queries before the next one, so the agent pays for a refresh once and serves the rest cheaply from the copy. After the final update only one query remains, so it rewrites again rather than refreshing.

Nobody encoded that rule. It falls out of the costs.

In [ ]:
policies = {
    'always rewrite': lambda s, acts: 'apply-update' if 'apply-update' in acts else 'serve:rewrite',
    'always from copy': lambda s, acts: 'apply-update' if 'apply-update' in acts else 'serve:materialised',
    'refresh every query': lambda s, acts: 'apply-update' if 'apply-update' in acts else 'serve:refresh-first',
    'optimal': mdp.greedy_policy(pi),
}
rows = []
for name, policy in policies.items():
    ep = mdp.run_episode(M, policy, max_steps=40)
    rows.append({'policy': name, 'return': round(ep.discounted_return(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print('\n"Always from copy" is the cheapest and by far the worst: it answers\n'
      'most queries from stale data and earns nothing for them.')

### Task 4.1 — Find the update rate that flips the strategy

Sweep workloads from update-heavy to read-heavy and report where the optimal policy stops refreshing at all, and where it starts serving mostly from the copy.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
updates = [r['updates'] for r in rows]
assert updates == sorted(updates, reverse=True), (
    'sweep from frequent updates down to none')
assert updates[0] > 0 and updates[-1] == 0
# Every query is served exactly one way.
assert all(r['refreshes'] + r['from copy'] + r['rewrites'] == r['workload'].count('q')
           for r in rows)
# The strategy flips: frequent updates favour rewriting, no updates favour the copy.
assert rows[0]['rewrites'] >= rows[-1]['rewrites']
assert rows[-1]['from copy'] > 0

### Task 4.2 — Remove the staleness penalty and watch the agent go wrong

Change the reward so a stale answer still scores `+1` — i.e. model latency only. Show the optimal policy becoming one that mostly serves wrong answers.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert stale_answers >= 4

### Task 4.3 — Give the agent a mapping it must verify

Use `compare_strategies` to show that a broken mapping is caught without any gold answer, and argue why this belongs in a tool rather than in the prompt.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert left != right

## Chapter 8 in the course arc

| | Ch. 5 | Ch. 6 | Ch. 8 |
|---|---|---|---|
| MDP | plan under prerequisites | diagnosis | **serve under staleness** |
| the error it prevents | ontologically wrong axiom | unsound part-whole chaining | **confident stale answers** |
| how it is caught | meta-property checker | relation taxonomy | **two execution paths disagreeing** |

Chapter 8's contribution to the course's argument about evaluation: the best checks need no gold answer. Running the same question two ways and demanding agreement catches a whole class of bug for free — and it is the pattern to reach for whenever you have two independent routes to the same result.